In [1]:
!pip install transformers medmnist scikit-learn torch torchvision scipy seaborn

# Notebook 3 — Feature Extraction & Model Training

Extracts **Phikon** (owkin/phikon) ViT embeddings for every patch in every bag,
then trains three models for ablation:

| # | Model | Modality | Architecture |
|---|---|---|---|
| **1** | WSI Only | Pathology image | Dual-stream ABMIL (attention + max-pool) |
| **2** | Molecular Only | Real TCGA RNA-seq (TCGA-COAD/READ) | 3-layer MLP |
| **3** | Multi-Modal THREADS | Both | WSI branch + Mol branch → cross-modal fusion |

Model 3 is a simplified implementation of the **THREADS** downstream inference
regime (Vaidya et al., 2024): the slide representation and the real molecular
expression profile are jointly used for the binary cancer / no-cancer decision.

> **Prerequisite:** Run `02_preprocessing_new.ipynb` first to generate `artifacts/`.

In [2]:
import numpy as np
import torch
import pickle, os

# ── Constants ─────────────────────────────────────────────────────────────────
NORMAL_LABEL  = 6
TUMOR_LABEL   = 8
BAG_SIZE      = 100
EMBEDDING_DIM = 768
N_TRAIN_BAGS  = 200
N_TEST_BAGS   = 60
ARTIFACTS_DIR = "artifacts"

# ── Load preprocessed artifacts ───────────────────────────────────────────────
with open(os.path.join(ARTIFACTS_DIR, "preprocessing.pkl"), "rb") as f:
    art = pickle.load(f)

train_bags       = art["train_bags"]
train_bag_labels = art["train_bag_labels"]
test_bags        = art["test_bags"]
test_bag_labels  = art["test_bag_labels"]
mol_train        = art["mol_train"]          # (200, 40) real TCGA log2(TPM+1)
mol_test         = art["mol_test"]           # ( 60, 40) real TCGA log2(TPM+1)
cancer_genes     = art["cancer_genes"]       # list of 40 gene names
MOLECULAR_DIM    = int(art["MOLECULAR_DIM"]) # loaded from artifact -- 40 real genes

mol_train_tensor = torch.tensor(mol_train, dtype=torch.float32)
mol_test_tensor  = torch.tensor(mol_test,  dtype=torch.float32)

print(f"Loaded {len(train_bags)} train bags and {len(test_bags)} test bags.")
print(f"mol_train : {mol_train.shape}   mol_test : {mol_test.shape}")
print(f"MOLECULAR_DIM = {MOLECULAR_DIM}  ({cancer_genes[:3]} ... {cancer_genes[-1]})")

Loaded 200 train bags and 60 test bags.
mol_train : (200, 40)   mol_test : (60, 40)
MOLECULAR_DIM = 40  (['KRAS', 'BRAF', 'MYC'] ... SFRP1)


In [3]:
import torch
import torchvision.transforms as transforms
from transformers import ViTModel
import medmnist
from medmnist import INFO
import gc

torch.cuda.empty_cache()
gc.collect()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

info      = INFO['pathmnist']
DataClass = getattr(medmnist, info['python_class'])
train_data = DataClass(split='train', download=True)
test_data  = DataClass(split='test',  download=True)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Loading Phikon (owkin/phikon) pathology foundation model ...")
phikon = ViTModel.from_pretrained("owkin/phikon", add_pooling_layer=False).to(device)
phikon.eval()

def extract_bag_features(bags, dataset, encoder, device, transform):
    '''Returns (N_bags, BAG_SIZE, 768) Phikon CLS-token tensor.'''
    feats = []
    with torch.no_grad():
        for bag_idx in bags:
            imgs = [transform(dataset[idx][0]) for idx in bag_idx]
            out  = encoder(torch.stack(imgs).to(device))
            feats.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.stack(feats)

print("\nExtracting train features (several minutes on CPU) ...")
X_train_tensor = extract_bag_features(train_bags, train_data, phikon, device, transform)
y_train_tensor = torch.tensor(train_bag_labels).view(-1, 1)

print("Extracting test features ...")
X_test_tensor  = extract_bag_features(test_bags,  test_data,  phikon, device, transform)
y_test_tensor  = torch.tensor(test_bag_labels).view(-1, 1)

print(f"\nX_train : {tuple(X_train_tensor.shape)}")
print(f"X_test  : {tuple(X_test_tensor.shape)}")

# ── Embedding verification ─────────────────────────────────────────────────────
emb_checks = {
    f"Train shape ({N_TRAIN_BAGS},{BAG_SIZE},{EMBEDDING_DIM})" :
        tuple(X_train_tensor.shape) == (N_TRAIN_BAGS, BAG_SIZE, EMBEDDING_DIM),
    "Train No NaN" : not torch.isnan(X_train_tensor).any().item(),
    "Train No Inf" : not torch.isinf(X_train_tensor).any().item(),
    f"Test  shape ({N_TEST_BAGS},{BAG_SIZE},{EMBEDDING_DIM})" :
        tuple(X_test_tensor.shape)  == (N_TEST_BAGS,  BAG_SIZE, EMBEDDING_DIM),
    "Test  No NaN" : not torch.isnan(X_test_tensor).any().item(),
    "Test  No Inf" : not torch.isinf(X_test_tensor).any().item(),
}
for k, v in emb_checks.items():
    print(f"  {k:<55s}  {'PASSED' if v else 'FAILED'}")

# Save for downstream notebooks
torch.save(X_train_tensor, os.path.join(ARTIFACTS_DIR, "X_train.pt"))
torch.save(X_test_tensor,  os.path.join(ARTIFACTS_DIR, "X_test.pt"))
torch.save(y_train_tensor, os.path.join(ARTIFACTS_DIR, "y_train.pt"))
torch.save(y_test_tensor,  os.path.join(ARTIFACTS_DIR, "y_test.pt"))
print("\nEmbeddings saved to artifacts/")


Device: cuda


Loading Phikon (owkin/phikon) pathology foundation model ...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: owkin/phikon
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Extracting train features (several minutes on CPU) ...


Extracting test features ...



X_train : (200, 100, 768)
X_test  : (60, 100, 768)
  Train shape (200,100,768)                                PASSED
  Train No NaN                                             PASSED
  Train No Inf                                             PASSED
  Test  shape (60,100,768)                                 PASSED
  Test  No NaN                                             PASSED
  Test  No Inf                                             PASSED

Embeddings saved to artifacts/


In [4]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (roc_auc_score, accuracy_score, roc_curve,
                              precision_score, recall_score,
                              average_precision_score)
import numpy as np

def evaluate(preds, labels, threshold=None):
    '''AUC, Average Precision, and optimal-threshold classification metrics.'''
    auc = roc_auc_score(labels, preds)
    ap  = average_precision_score(labels, preds)
    fpr, tpr, thrs = roc_curve(labels, preds)
    if threshold is None:
        idx       = np.argmax(tpr - fpr)
        threshold = float(thrs[idx])
    binary = [1 if p >= threshold else 0 for p in preds]
    return dict(preds=preds, labels=labels, auc=auc, ap=ap, threshold=threshold,
                acc=accuracy_score(labels, binary),
                precision=precision_score(labels, binary, zero_division=0),
                recall=recall_score(labels, binary, zero_division=0),
                fpr=fpr, tpr=tpr)

loss_fn = nn.BCELoss()


---
## Model 1 — WSI Only: Dual-Stream ABMIL

Implements the WSI branch of THREADS:
Phikon patch embeddings → **Attention pooling** + **Max pooling** → concatenate → MLP classifier.

The dual-stream design ensures both the _contextually relevant_ patches
(attention) and the single _strongest anomaly_ (max-pool) are captured.

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 1 — WSI Only: Dual-Stream ABMIL
# ═══════════════════════════════════════════════════════════════════════════════
class THREADS_WSI_Only(nn.Module):
    '''
    Attention-Based MIL with dual aggregation (attention + max-pool).
    WSI branch of the THREADS architecture (Vaidya et al., 2024).
    '''
    def __init__(self, input_dim=768, hidden_dim=512):
        super().__init__()
        self.pre   = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(0.2))
        self.att_a = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.Dropout(0.2))
        self.att_b = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.Dropout(0.2))
        self.att_c = nn.Linear(hidden_dim, 1)
        self.post  = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(0.2))
        self.clf   = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h     = self.pre(x.squeeze(0))
        a     = torch.tanh(self.att_a(h))
        b     = torch.sigmoid(self.att_b(h))
        A     = F.softmax(self.att_c(a * b), dim=0)
        M_att = torch.mm(A.t(), h)
        M_max = torch.max(h, dim=0, keepdim=True)[0]
        M     = self.post(torch.cat([M_att, M_max], dim=1))
        return torch.sigmoid(self.clf(M)), A

model_wsi = THREADS_WSI_Only(input_dim=EMBEDDING_DIM, hidden_dim=512).to(device)
opt_wsi   = torch.optim.Adam(model_wsi.parameters(), lr=1e-4, weight_decay=1e-4)
EPOCHS_1  = 40

print("Training Model 1 — WSI Only (Dual-Stream ABMIL) ...")
for epoch in range(EPOCHS_1):
    model_wsi.train()
    total_loss = 0.0
    for idx in torch.randperm(len(X_train_tensor)):
        bag   = X_train_tensor[idx].unsqueeze(0).to(device)
        label = y_train_tensor[idx].to(device)
        opt_wsi.zero_grad()
        pred, _ = model_wsi(bag)
        loss = loss_fn(pred.squeeze(), label.squeeze())
        loss.backward()
        opt_wsi.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS_1}  Loss: {total_loss/N_TRAIN_BAGS:.4f}")

# Quick eval
model_wsi.eval()
wsi_preds = []
with torch.no_grad():
    for i in range(len(X_test_tensor)):
        prob, _ = model_wsi(X_test_tensor[i].unsqueeze(0).to(device))
        wsi_preds.append(prob.item())

wsi_labels  = [y_test_tensor[i].item() for i in range(len(y_test_tensor))]
results_wsi = evaluate(wsi_preds, wsi_labels)
print(f"\n  Model 1 (WSI Only)  AUC={results_wsi['auc']:.4f}  "
      f"AP={results_wsi['ap']:.4f}  Acc={results_wsi['acc']*100:.2f}%")

torch.save(model_wsi.state_dict(), os.path.join(ARTIFACTS_DIR, "model_wsi.pt"))
print("  Saved: artifacts/model_wsi.pt")


Training Model 1 — WSI Only (Dual-Stream ABMIL) ...


  Epoch 10/40  Loss: 0.1146


  Epoch 20/40  Loss: 0.0007


  Epoch 30/40  Loss: 0.0017


  Epoch 40/40  Loss: 0.0058

  Model 1 (WSI Only)  AUC=0.8356  AP=0.8727  Acc=78.33%
  Saved: artifacts/model_wsi.pt


---
## Model 2 — Molecular Only: MLP Baseline

Ablation baseline that uses **only** the real TCGA RNA-seq molecular profile
(log2(TPM+1) normalised expression of 40 colorectal cancer driver genes).
Isolates the predictive power of the molecular modality in isolation.

In [6]:
from torch.utils.data import TensorDataset, DataLoader

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 2 — Molecular Only: MLP on real TCGA RNA-seq profiles
# ═══════════════════════════════════════════════════════════════════════════════
class MolecularOnly_Classifier(nn.Module):
    '''3-layer MLP on real TCGA log2(TPM+1) molecular features.'''
    def __init__(self, input_dim=40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.GELU(),           nn.Dropout(0.3),
            nn.Linear(64, 1))

    def forward(self, x):
        return torch.sigmoid(self.net(x)), None

model_mol = MolecularOnly_Classifier(input_dim=MOLECULAR_DIM).to(device)
opt_mol   = torch.optim.Adam(model_mol.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS_2  = 60

mol_loader = DataLoader(
    TensorDataset(mol_train_tensor, y_train_tensor), batch_size=8, shuffle=True)

print("Training Model 2 — Molecular Only (MLP on real TCGA RNA-seq) ...")
for epoch in range(EPOCHS_2):
    model_mol.train()
    total_loss = 0.0
    for feat, label in mol_loader:
        feat, label = feat.to(device), label.to(device)
        opt_mol.zero_grad()
        pred, _ = model_mol(feat)
        loss = loss_fn(pred.squeeze(), label.squeeze())
        loss.backward()
        opt_mol.step()
        total_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS_2}  Loss: {total_loss/len(mol_loader):.4f}")

model_mol.eval()
mol_preds = []
with torch.no_grad():
    for i in range(len(mol_test_tensor)):
        prob, _ = model_mol(mol_test_tensor[i].unsqueeze(0).to(device))
        mol_preds.append(prob.item())

results_mol = evaluate(mol_preds, wsi_labels)
print(f"\n  Model 2 (Mol Only)  AUC={results_mol['auc']:.4f}  "
      f"AP={results_mol['ap']:.4f}  Acc={results_mol['acc']*100:.2f}%")

torch.save(model_mol.state_dict(), os.path.join(ARTIFACTS_DIR, "model_mol.pt"))
print("  Saved: artifacts/model_mol.pt")

Training Model 2 — Molecular Only (MLP on real TCGA RNA-seq) ...


  Epoch 20/60  Loss: 0.0161


  Epoch 40/60  Loss: 0.0004


  Epoch 60/60  Loss: 0.0002

  Model 2 (Mol Only)  AUC=1.0000  AP=1.0000  Acc=100.00%
  Saved: artifacts/model_mol.pt


---
## Model 3 — Multi-Modal THREADS (WSI + Molecular)

Full implementation of the **THREADS** downstream inference regime:

```
WSI patches → Phikon → Dual-Stream ABMIL (attention + max-pool) → 512-dim
RNA-seq profile → MLP encoder → 128-dim
Concatenate → 640-dim → LayerNorm MLP → binary classifier
```

The molecular branch provides a complementary patient-level signal that guides
the WSI branch toward the tumour patch, even when the visual signal is ambiguous.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 3 — Multi-Modal THREADS (WSI + Real Molecular)
# ═══════════════════════════════════════════════════════════════════════════════
class THREADS_MultiModal(nn.Module):
    '''
    Multi-modal THREADS: dual-stream ABMIL (WSI) fused with MLP (real TCGA RNA-seq).
    Based on: Vaidya et al., Molecular-driven Foundation Model for Oncologic
    Pathology, 2024.
    '''
    def __init__(self, wsi_dim=768, mol_dim=40, hidden_dim=512, mol_hidden=128):
        super().__init__()
        # WSI stream
        self.wsi_pre  = nn.Sequential(
            nn.Linear(wsi_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(0.2))
        self.att_a    = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.Dropout(0.2))
        self.att_b    = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.Dropout(0.2))
        self.att_c    = nn.Linear(hidden_dim, 1)
        self.wsi_post = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(0.2))
        # Molecular stream (real TCGA log2(TPM+1) gene expression)
        self.mol_enc  = nn.Sequential(
            nn.Linear(mol_dim,    mol_hidden), nn.LayerNorm(mol_hidden),
            nn.GELU(), nn.Dropout(0.2),
            nn.Linear(mol_hidden, mol_hidden), nn.GELU())
        # Cross-modal fusion
        self.fusion   = nn.Sequential(
            nn.Linear(hidden_dim + mol_hidden, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(0.2))
        self.clf = nn.Linear(hidden_dim, 1)

    def forward(self, wsi_bag, mol_feat):
        h      = self.wsi_pre(wsi_bag.squeeze(0))
        a      = torch.tanh(self.att_a(h))
        b      = torch.sigmoid(self.att_b(h))
        A      = F.softmax(self.att_c(a * b), dim=0)
        M_wsi  = self.wsi_post(torch.cat([torch.mm(A.t(), h),
                                           torch.max(h, 0, keepdim=True)[0]], dim=1))
        mol    = mol_feat.unsqueeze(0) if mol_feat.dim() == 1 else mol_feat
        M_mol  = self.mol_enc(mol)
        M      = self.fusion(torch.cat([M_wsi, M_mol], dim=1))
        return torch.sigmoid(self.clf(M)), A

model_mm = THREADS_MultiModal(
    wsi_dim=EMBEDDING_DIM, mol_dim=MOLECULAR_DIM, hidden_dim=512).to(device)
opt_mm   = torch.optim.Adam(model_mm.parameters(), lr=1e-4, weight_decay=1e-4)
EPOCHS_3 = 40

print("Training Model 3 — Multi-Modal THREADS (WSI + Real TCGA RNA-seq) ...")
for epoch in range(EPOCHS_3):
    model_mm.train()
    total_loss = 0.0
    for idx in torch.randperm(len(X_train_tensor)):
        bag   = X_train_tensor[idx].unsqueeze(0).to(device)
        mol   = mol_train_tensor[idx].to(device)
        label = y_train_tensor[idx].to(device)
        opt_mm.zero_grad()
        pred, _ = model_mm(bag, mol)
        loss = loss_fn(pred.squeeze(), label.squeeze())
        loss.backward()
        opt_mm.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS_3}  Loss: {total_loss/N_TRAIN_BAGS:.4f}")

model_mm.eval()
mm_preds = []
with torch.no_grad():
    for i in range(len(X_test_tensor)):
        bag  = X_test_tensor[i].unsqueeze(0).to(device)
        mol  = mol_test_tensor[i].to(device)
        prob, _ = model_mm(bag, mol)
        mm_preds.append(prob.item())

mm_labels  = wsi_labels
results_mm = evaluate(mm_preds, mm_labels)
print(f"\n  Model 3 (Multi-Modal)  AUC={results_mm['auc']:.4f}  "
      f"AP={results_mm['ap']:.4f}  Acc={results_mm['acc']*100:.2f}%")

torch.save(model_mm.state_dict(), os.path.join(ARTIFACTS_DIR, "model_mm.pt"))
print("  Saved: artifacts/model_mm.pt")

Training Model 3 — Multi-Modal THREADS (WSI + Real TCGA RNA-seq) ...


  Epoch 10/40  Loss: 0.0625


  Epoch 20/40  Loss: 0.0248


  Epoch 30/40  Loss: 0.0009


  Epoch 40/40  Loss: 0.0005

  Model 3 (Multi-Modal)  AUC=1.0000  AP=1.0000  Acc=100.00%
  Saved: artifacts/model_mm.pt


In [8]:
import pickle

results = {
    "results_wsi" : results_wsi,
    "results_mol" : results_mol,
    "results_mm"  : results_mm,
    "wsi_labels"  : wsi_labels,
    "mm_labels"   : mm_labels,
}
with open(os.path.join(ARTIFACTS_DIR, "results.pkl"), "wb") as f:
    pickle.dump(results, f)

print("All models and results saved.")
print("Run Notebook 04 for full evaluation.")


All models and results saved.
Run Notebook 04 for full evaluation.
